In [28]:
import os
from typing import TypedDict, Annotated
import operator
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [21]:
load_dotenv()

True

In [24]:
class ScoreModel(BaseModel):
    feedback: str = Field(description="Feedback for evaluation criterion")
    score : int = Field(description="Score out of 10 for evaluation criterion", ge=0, le=10)

In [ ]:
class Essay(TypedDict):
    essay: str 
    feedback1: str
    feedback2: str
    feedback3: str
    score : Annotated[list[int], operator.add]

In [30]:
essay = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development."""


In [34]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.environ.get("gemini_apikey"))
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

In [35]:
llm = model.with_structured_output(ScoreModel)

In [36]:
def create_essay(state: Essay):
    # prompt = "create an essay of 100 words on the topic: " + state["topic"]
    # essay = llm.invoke(prompt)
    # state["essay"] = essay.content
    return state

In [37]:
# depth of analysis
def check_depth(state:Essay):
    prompt = "check the depth of analysis in the essay and give a score out of 10 and text feedback"
    response = llm.invoke(prompt + "\n\n" + state["essay"])
    # Assuming the response is a string with the score and feedback separated by a newline
    score, feedback = response.content.split("\n", 1)

    return { "score1": (score), "feedback1": feedback }

In [38]:
# clarity of thought
def check_clarity(state:Essay):
    prompt = "check the clarity of thought in the essay and give a score out of 10 and text feedback"
    response = llm.invoke(prompt + "\n\n" + state["essay"])
    score, feedback = response.content.split("\n", 1)
    return { "score2": (score), "feedback2": feedback }

# language and style
def check_language(state:Essay):
    prompt = "check the language and style in the essay and give a score out of 10 and text feedback"
    response = llm.invoke(prompt + "\n\n" + state["essay"])
    score, feedback = response.content.split("\n", 1)

    return { "score3": (score), "feedback3": feedback }

In [39]:
def summary(state:Essay):
    total_score = state["score1"] + state["score2"] + state["score3"]
    average_score = total_score
    summary_text = f"Total Score: {total_score}, Average Score: {average_score}\n"
    summary_text += f"Depth Feedback: {state['feedback1']}\n"
    summary_text += f"Clarity Feedback: {state['feedback2']}\n"
    summary_text += f"Language Feedback: {state['feedback3']}\n"
    
    return {"summary": summary_text}

In [40]:
graph = StateGraph(Essay)

graph.add_node("create_essay", create_essay)
graph.add_node("check_depth", check_depth)
graph.add_node("check_clarity", check_clarity)
graph.add_node("check_language", check_language)
graph.add_node("summary", summary)

# create edges
graph.add_edge(START, "create_essay")
graph.add_edge("create_essay", "check_depth")
graph.add_edge("create_essay", "check_clarity")
graph.add_edge("create_essay", "check_language")
graph.add_edge("check_depth", "summary")
graph.add_edge("check_clarity", "summary")
graph.add_edge("check_language", "summary")
graph.add_edge("summary", END)

# compile graph
app = graph.compile()

In [15]:
initial_state = {'topic': 'What is LLM?'}
final_state = app.invoke(initial_state)

In [16]:
print(final_state)

{'topic': 'What is LLM?', 'essay': 'Large Language Models (LLMs) are advanced artificial‑intelligence systems that generate human‑like text. Trained on massive corpora of books, articles, and web content, they learn statistical patterns of language, enabling them to predict the next word in a sequence. This predictive ability fuels applications such as chatbots, translation tools, and content creation assistants. LLMs rely on deep neural architectures, typically transformers, which capture contextual relationships across long passages. While they excel at fluency and knowledge recall, they lack true understanding and can produce plausible‑but‑incorrect statements. Ongoing research focuses on improving reliability, interpretability, and ethical deployment of these powerful models in society.', 'feedback1': '\n**Feedback**\n\n| Aspect | Evaluation | Comments |\n|--------|------------|----------|\n| **Breadth of coverage** | **Good** (3/3) | The essay mentions what LLMs are, how they’re t

In [17]:
llm.invoke("What is the api key?")

AIMessage(content='I’m sorry, but I can’t help with that.', additional_kwargs={'reasoning_content': 'The user asks "What is the api key?" This is a request for disallowed content: providing API keys. According to policy, we must refuse. The user is asking for an API key, presumably for OpenAI. Must refuse.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 77, 'total_tokens': 147, 'completion_time': 0.144889857, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.002866645, 'prompt_tokens_details': None, 'queue_time': 0.36836222, 'total_time': 0.147756502}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fffb5-9cd3-7bf1-ac47-4393458bba85-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 70, 'total_tokens': 147, 'output_token_details': {'reasoning': 49}})

In [18]:
from IPython.display import display, Markdown

# Define your Markdown string
# markdown_text = "### Success!\nThis text is rendered using **Markdown syntax** inside a notebook cell."

# Render it on screen
for i in final_state.keys():
    display(Markdown(f"{i}: {final_state[i]}"))

topic: What is LLM?

essay: Large Language Models (LLMs) are advanced artificial‑intelligence systems that generate human‑like text. Trained on massive corpora of books, articles, and web content, they learn statistical patterns of language, enabling them to predict the next word in a sequence. This predictive ability fuels applications such as chatbots, translation tools, and content creation assistants. LLMs rely on deep neural architectures, typically transformers, which capture contextual relationships across long passages. While they excel at fluency and knowledge recall, they lack true understanding and can produce plausible‑but‑incorrect statements. Ongoing research focuses on improving reliability, interpretability, and ethical deployment of these powerful models in society.

feedback1: 
**Feedback**

| Aspect | Evaluation | Comments |
|--------|------------|----------|
| **Breadth of coverage** | **Good** (3/3) | The essay mentions what LLMs are, how they’re trained, the core architecture (transformers), key applications, and major limitations (lack of true understanding, hallucinations). |
| **Depth of explanation** | **Limited** (1/3) | The description stays at a high‑level overview. It does not explain *how* transformers capture context (e.g., self‑attention), the role of pre‑training vs. fine‑tuning, or the statistical learning mechanisms (e.g., maximum likelihood). |
| **Critical analysis** | **Weak** (0/2) | The essay notes that LLMs “lack true understanding” and can produce “plausible‑but‑incorrect statements,” but it does not explore why these failures occur, what metrics reveal them, or how they compare to human cognition. |
| **Evidence & examples** | **Sparse** (0/1) | No concrete studies, benchmark results, or real‑world incidents are cited to substantiate claims about reliability, interpretability, or ethical concerns. |
| **Discussion of research directions** | **Superficial** (0/1) | “Improving reliability, interpretability, and ethical deployment” is mentioned, but the essay does not detail current approaches (e.g., RLHF, model‑editing, provenance tracking) or open challenges. |
| **Structure & coherence** | **Adequate** (1/1) | The paragraph flows logically from definition to capabilities, limitations, and future work. |

### Overall Assessment
The essay provides a solid **introductory snapshot** of large language models but stops short of a **deep analytical treatment**. It would benefit from:

1. **Technical depth** – explain key mechanisms (self‑attention, token embeddings, scaling laws) and how they enable the observed capabilities.
2. **Critical evaluation** – dissect specific failure modes (e.g., factual hallucination, bias amplification) and discuss underlying causes.
3. **Empirical support** – reference benchmark scores (e.g., GLUE, MMLU) or notable case studies that illustrate strengths and weaknesses.
4. **Research frontiers** – elaborate on concrete methods being pursued (e.g., retrieval‑augmented generation, interpretability tools, alignment techniques) and the open questions they aim to solve.

Adding these elements would raise the analytical depth substantially, moving the score toward the upper half of the scale.

feedback2: 
### What works well
1. **Logical flow** – The paragraph moves smoothly from a definition of LLMs, to how they are trained, to their core technology (transformers), then to strengths, limitations, and finally to research directions.  
2. **Concise terminology** – Key concepts (“predict the next word,” “deep neural architectures,” “transformers”) are introduced briefly but accurately, keeping the piece readable for a general audience.  
3. **Balanced perspective** – You acknowledge both the impressive capabilities (fluency, knowledge recall) and the important shortcomings (lack of true understanding, plausible‑but‑incorrect output), which adds credibility.

### Areas for improvement
| Issue | Why it matters | Suggested tweak |
|-------|----------------|-----------------|
| **Sentence density** | Some sentences pack multiple ideas, which can make the reader pause to unpack them. | Split the second sentence: “They are trained on massive corpora of books, articles, and web content. By learning statistical patterns of language, they can predict the next word in a sequence.” |
| **Transition between ideas** | The jump from “transformers” to “while they excel…” feels abrupt. | Add a brief bridge, e.g., “Because transformers capture long‑range context, LLMs achieve remarkable fluency.” |
| **Specificity of “research focuses”** | The final clause is vague; naming a couple of concrete research themes would sharpen the conclusion. | Example: “Current research aims to improve reliability (e.g., factual grounding), enhance interpretability (e.g., attention‑visualization tools), and address ethical concerns such as bias and misuse.” |
| **Repetition of “powerful models”** | The phrase “powerful models” repeats the idea already conveyed by “advanced AI systems.” | Consider dropping “powerful” or rephrasing: “…these models in society.” |
| **Citation of examples** | Mentioning a concrete chatbot or translation tool would ground the abstract description. | E.g., “applications such as ChatGPT, Google Translate, and automated content‑creation assistants.” |

### Overall impression
Your essay presents a clear, well‑structured overview of large language models. The main ideas are easy to follow, and the balance between technical description and high‑level implications is appropriate for a short introductory piece. Minor adjustments to sentence structure, transitions, and specificity would raise the clarity further, pushing the score toward a perfect 10.

feedback3: 
### What works well
1. **Clarity & Conciseness** – The essay conveys the core ideas about LLMs in a straightforward, compact manner. Each sentence adds a distinct piece of information without unnecessary filler.
2. **Logical Flow** – The progression from definition → training data → architecture → strengths → limitations → research directions is logical and easy to follow.
3. **Technical Accuracy** – Key terms (e.g., “transformers,” “statistical patterns,” “interpretability”) are used correctly, showing a solid grasp of the subject.
4. **Balanced Tone** – The writing maintains an objective, academic tone appropriate for an overview article.

### Areas for improvement
| Aspect | Issue | Suggested tweak |
|--------|-------|-----------------|
| **Sentence variety** | Several sentences start with “they” or “LLMs,” which can feel repetitive. | Vary the subject or use transitional phrases (e.g., “Consequently, they excel…”, “Because of this architecture, LLMs can…”). |
| **Specificity** | “Massive corpora” and “powerful models” are vague. | Add a rough scale or example (e.g., “trained on billions of tokens” or “models with hundreds of billions of parameters”). |
| **Parallel structure** | The list of applications (“chatbots, translation tools, and content creation assistants”) mixes nouns and a noun phrase. | Keep parallelism: “chatbots, translation systems, and content‑creation assistants.” |
| **Citation of challenges** | “Plausible‑but‑incorrect statements” is a good point but could be framed as a known term (“hallucinations”). | Replace with “they can hallucinate—producing plausible‑but‑incorrect statements.” |
| **Transition to research** | The shift to “Ongoing research” feels abrupt. | Add a bridging clause: “Given these limitations, ongoing research focuses on…” |
| **Stylistic polish** | Use of hyphenated compounds (“human‑like,” “plausible‑but‑incorrect”) is correct, but consistency matters. | Ensure all hyphenations follow the same style (e.g., “human‑like” and “plausible‑but‑incorrect” are fine). |
| **Conclusion** | The essay ends on a research note but lacks a closing sentence that ties back to the opening definition. | Add a brief concluding line, e.g., “Thus, while LLMs have transformed how machines generate language, ensuring their reliability remains a central challenge.” |

### Revised excerpt (illustrating the suggestions)

> Large Language Models (LLMs) are advanced artificial‑intelligence systems that generate human‑like text. Trained on billions of tokens drawn from books, articles, and web content, they learn statistical language patterns that enable next‑word prediction. This capability powers applications such as chatbots, translation systems, and content‑creation assistants. LLMs rely on deep neural architectures—most commonly transformers—that capture contextual relationships across long passages. Although they excel at fluency and knowledge recall, they can hallucinate, producing plausible‑but‑incorrect statements. Given these limitations, ongoing research focuses on improving reliability, interpretability, and the ethical deployment of these powerful models in society. Thus, while LLMs have reshaped automated language generation, ensuring their trustworthiness remains a central challenge.

Overall, the essay is solid and informative; polishing the points above will raise it from good to excellent.

score1: **Depth‑of‑Analysis Score: 4 / 10**

score2: **Clarity‑of‑Thought Score: 8 / 10**

score3: **Score: 8 / 10**